# 05 - Modelado

Baseline histórico, Regresión Lineal y Random Forest / Gradient Boosting para predecir `generacion_fotovoltaica` del día siguiente, con validación temporal y comparación sistemática frente al baseline.

## 1. Carga de los splits

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

def evaluar(y_true, y_pred, nombre):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    print(f"{nombre:25s} MAE = {mae:10.2f}   RMSE = {rmse:10.2f}")
    return {"modelo": nombre, "MAE": mae, "RMSE": rmse}

train = pd.read_parquet("../data/gold/train.parquet")
val = pd.read_parquet("../data/gold/val.parquet")
test = pd.read_parquet("../data/gold/test.parquet")

print("Train:", train.shape, train["fecha"].min(), "-", train["fecha"].max())
print("Val:  ", val.shape, val["fecha"].min(), "-", val["fecha"].max())
print("Test: ", test.shape, test["fecha"].min(), "-", test["fecha"].max())

Train: (1087, 29) 2021-01-09 00:00:00 - 2023-12-31 00:00:00
Val:   (366, 29) 2024-01-01 00:00:00 - 2024-12-31 00:00:00
Test:  (365, 29) 2025-01-01 00:00:00 - 2025-12-31 00:00:00


## 2. Selección de features

**Importante:** se excluyen las variables meteorológicas del propio día (`tmed`, `prec`, `velmedia`, ...) porque son observaciones reales, no un forecast conocido en el momento de predicción — usarlas causaría leakage (ver nota en `04_feature_engineering`). Se usan en su lugar las versiones `_lag_1` de esas variables, además de los lags y la media móvil de generación, y las variables temporales de calendario.

In [3]:
variables_meteo_base = [
    "tmed", "tmin", "tmax", "prec",
    "velmedia", "racha",
    "hrMedia", "presMax", "presMin",
    "sol"
]

features = (
    ["generacion_lag_1", "generacion_lag_7", "generacion_media_7d"]
    + [f"{v}_lag_1" for v in variables_meteo_base]
    + ["dia_semana", "mes", "dia_año"]
)

target = "generacion_fotovoltaica"

# Verificacion: todas las features deben existir en el dataset
faltan = [f for f in features if f not in train.columns]
assert len(faltan) == 0, f"Faltan columnas: {faltan}"

print("Numero de features:", len(features))
print(features)

Numero de features: 16
['generacion_lag_1', 'generacion_lag_7', 'generacion_media_7d', 'tmed_lag_1', 'tmin_lag_1', 'tmax_lag_1', 'prec_lag_1', 'velmedia_lag_1', 'racha_lag_1', 'hrMedia_lag_1', 'presMax_lag_1', 'presMin_lag_1', 'sol_lag_1', 'dia_semana', 'mes', 'dia_año']


In [4]:
X_train, y_train = train[features], train[target]
X_val, y_val = val[features], val[target]
X_test, y_test = test[features], test[target]

# Los NaN residuales (si los hubiera) en features de validacion/test se
# imputan con la mediana de train, nunca calculada sobre val/test
medianas_train = X_train.median()

X_train = X_train.fillna(medianas_train)
X_val = X_val.fillna(medianas_train)
X_test = X_test.fillna(medianas_train)

print("Nulos en X_train tras imputar:", X_train.isna().sum().sum())
print("Nulos en X_val tras imputar:", X_val.isna().sum().sum())
print("Nulos en X_test tras imputar:", X_test.isna().sum().sum())

Nulos en X_train tras imputar: 0
Nulos en X_val tras imputar: 0
Nulos en X_test tras imputar: 0


## 3. Baseline histórico

Predicción = generación del día anterior (`generacion_lag_1`). Cualquier modelo de Machine Learning debe superar esto de forma consistente para justificar su uso.

In [5]:
def evaluar(y_true, y_pred, nombre):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    print(f"{nombre:25s} MAE = {mae:10.2f}   RMSE = {rmse:10.2f}")
    return {"modelo": nombre, "MAE": mae, "RMSE": rmse}

resultados = []

pred_baseline_val = X_val["generacion_lag_1"]
resultados.append(evaluar(y_val, pred_baseline_val, "Baseline (val)"))

Baseline (val)            MAE =   17277.31   RMSE =   22631.86


## 4. Regresión Lineal

In [6]:
modelo_lr = LinearRegression()
modelo_lr.fit(X_train, y_train)

pred_lr_val = modelo_lr.predict(X_val)
resultados.append(evaluar(y_val, pred_lr_val, "Regresion Lineal (val)"))

Regresion Lineal (val)    MAE =   15868.69   RMSE =   20399.55


## 5. Random Forest

In [7]:
modelo_rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
modelo_rf.fit(X_train, y_train)

pred_rf_val = modelo_rf.predict(X_val)
resultados.append(evaluar(y_val, pred_rf_val, "Random Forest (val)"))

Random Forest (val)       MAE =   22357.95   RMSE =   27788.86


## 6. Gradient Boosting

In [8]:
modelo_gb = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)
modelo_gb.fit(X_train, y_train)

pred_gb_val = modelo_gb.predict(X_val)
resultados.append(evaluar(y_val, pred_gb_val, "Gradient Boosting (val)"))

Gradient Boosting (val)   MAE =   26656.47   RMSE =   33401.58


## 7. Comparación de modelos en validación

In [9]:
tabla_resultados = pd.DataFrame(resultados).sort_values("MAE")
tabla_resultados["mejora_vs_baseline_%"] = (
    (tabla_resultados.loc[tabla_resultados["modelo"] == "Baseline (val)", "MAE"].values[0]
    - tabla_resultados["MAE"]) /
    tabla_resultados.loc[tabla_resultados["modelo"] == "Baseline (val)", "MAE"].values[0] * 100
)

tabla_resultados

,modelo,MAE,RMSE,mejora_vs_baseline_%
1,Regresion Lineal (val),15868.692409,20399.548547,8.152976
0,Baseline (val),17277.307156,22631.857429,0.000000
2,Random Forest (val),22357.945434,27788.857778,-29.406424
3,Gradient Boosting (val),26656.469327,33401.584639,-54.286019


## 8. Análisis de errores por mes

Además de la métrica global, se revisa si el modelo elegido tiene un comportamiento estable a lo largo del año o si empeora en periodos concretos (relevante de cara al análisis de resiliencia posterior).

In [10]:
# Se usa el modelo que realmente supera al baseline en validacion (ver seccion 7)
modelo_elegido = modelo_lr
pred_val_elegido = modelo_elegido.predict(X_val)

errores_val = val[["fecha", "mes", target]].copy()
errores_val["prediccion"] = pred_val_elegido
errores_val["error_abs"] = (errores_val[target] - errores_val["prediccion"]).abs()

errores_por_mes = (
    errores_val.groupby("mes")["error_abs"]
    .agg(["mean", "std", "count"])
    .rename(columns={"mean": "MAE_mes", "std": "std_error", "count": "dias"})
)

errores_por_mes

,MAE_mes,std_error,dias
mes,,,
1,14021.544090,9405.435351,31
2,19714.021974,14318.667017,29
3,20383.841940,16487.637712,31
4,15552.028032,12538.638775,30
5,17035.065366,10824.449783,31
6,20549.708712,13378.712413,30
7,16428.031356,12466.923481,31
8,9740.140411,9375.747677,31
9,13594.818667,11611.024674,30


## 9. Backtesting temporal

Se repite el entrenamiento y evaluación en varias ventanas desplazadas en el tiempo (usando train+val como conjunto disponible, sin tocar test todavía) para comprobar si el modelo elegido tiene un rendimiento estable o varía mucho según el periodo. Cada ventana entrena solo con datos anteriores a su propio periodo de evaluación, respetando el orden temporal.

In [11]:
# Conjunto disponible para backtesting: train + val (test se reserva para el final)
disponible = pd.concat([train, val]).sort_values("fecha").reset_index(drop=True)

# Ventanas de evaluacion de ~2 meses cada una, deslizando a lo largo del periodo
fechas_corte = pd.date_range(
    disponible["fecha"].min() + pd.Timedelta(days=365),
    disponible["fecha"].max() - pd.Timedelta(days=60),
    freq="60D"
)

resultados_backtest = []

for corte in fechas_corte:
    fin_ventana = corte + pd.Timedelta(days=60)

    train_bt = disponible[disponible["fecha"] < corte]
    test_bt = disponible[(disponible["fecha"] >= corte) & (disponible["fecha"] < fin_ventana)]

    if len(train_bt) < 90 or len(test_bt) == 0:
        continue

    X_train_bt = train_bt[features].fillna(train_bt[features].median())
    y_train_bt = train_bt[target]
    X_test_bt = test_bt[features].fillna(train_bt[features].median())
    y_test_bt = test_bt[target]

    modelo_bt = LinearRegression()
    modelo_bt.fit(X_train_bt, y_train_bt)
    pred_bt = modelo_bt.predict(X_test_bt)

    mae_bt = mean_absolute_error(y_test_bt, pred_bt)
    mae_baseline_bt = mean_absolute_error(y_test_bt, X_test_bt["generacion_lag_1"])

    resultados_backtest.append({
        "ventana_inicio": corte.date(),
        "ventana_fin": fin_ventana.date(),
        "dias_train": len(train_bt),
        "dias_test": len(test_bt),
        "MAE_modelo": mae_bt,
        "MAE_baseline": mae_baseline_bt,
        "mejora_%": (mae_baseline_bt - mae_bt) / mae_baseline_bt * 100
    })

tabla_backtest = pd.DataFrame(resultados_backtest)
tabla_backtest

,ventana_inicio,ventana_fin,dias_train,dias_test,MAE_modelo,MAE_baseline,mejora_%
0,2022-01-09,2022-03-10,365,60,8473.566718,8612.497233,1.613127
1,2022-03-10,2022-05-09,425,60,17889.031275,18761.793433,4.651806
2,2022-05-09,2022-07-08,485,60,9127.926694,10018.550667,8.889749
3,2022-07-08,2022-09-06,545,60,5900.736508,6348.436017,7.052123
4,2022-09-06,2022-11-05,605,60,11926.509121,13438.111900,11.248625
5,2022-11-05,2023-01-04,665,60,9976.954221,10701.783300,6.772975
6,2023-01-04,2023-03-05,725,60,9640.242283,9930.217383,2.920128
7,2023-03-05,2023-05-04,785,60,13179.150552,14348.941783,8.152456
8,2023-05-04,2023-07-03,845,60,13745.257965,13850.439633,0.759410
9,2023-07-03,2023-09-01,905,60,7645.021330,8805.857950,13.182550


In [12]:
print("Ventanas evaluadas:", len(tabla_backtest))
print("Ventanas donde el modelo supera al baseline:",
    (tabla_backtest["mejora_%"] > 0).sum(), "/", len(tabla_backtest))
print()
print("MAE modelo   -> media:", tabla_backtest["MAE_modelo"].mean().round(2),
    "| std:", tabla_backtest["MAE_modelo"].std().round(2))
print("MAE baseline -> media:", tabla_backtest["MAE_baseline"].mean().round(2),
    "| std:", tabla_backtest["MAE_baseline"].std().round(2))
print()
print("Media de mejora: {:.2f}%".format(tabla_backtest["mejora_%"].mean()))
print("Mediana de mejora: {:.2f}%".format(tabla_backtest["mejora_%"].median()))
print("Desviacion tipica de la mejora: {:.2f}".format(tabla_backtest["mejora_%"].std()))
print("Mejora minima en una ventana: {:.2f}%".format(tabla_backtest["mejora_%"].min()))
print("Mejora maxima en una ventana: {:.2f}%".format(tabla_backtest["mejora_%"].max()))

Ventanas evaluadas: 18
Ventanas donde el modelo supera al baseline: 18 / 18

MAE modelo   -> media: 12771.83 | std: 3795.73
MAE baseline -> media: 13796.69 | std: 4190.92

Media de mejora: 7.18%
Mediana de mejora: 6.91%
Desviacion tipica de la mejora: 4.20
Mejora minima en una ventana: 0.76%
Mejora maxima en una ventana: 15.30%


## 10. Evaluación final en test

Se ejecuta una única vez, con el modelo ya seleccionado según el rendimiento en validación — el conjunto de test no se usa para decidir entre modelos.

In [13]:
pred_test = modelo_elegido.predict(X_test)
pred_baseline_test = X_test["generacion_lag_1"]

resultado_test_modelo = evaluar(y_test, pred_test, "Modelo elegido (test)")
resultado_test_baseline = evaluar(y_test, pred_baseline_test, "Baseline (test)")

Modelo elegido (test)     MAE =   18640.39   RMSE =   23499.25
Baseline (test)           MAE =   20983.37   RMSE =   26255.02


## 11. Tabla de salida (formato de la capa de predicciones)

In [14]:
salida = pd.DataFrame({
    "fecha": test["fecha"].values,
    "generacion_real": y_test.values,
    "generacion_predicha": pred_test,
    "error": y_test.values - pred_test,
    "modelo": "RegresionLineal"  # ajustar si cambia el modelo elegido
})

salida.to_csv(
    "../data/gold/predicciones_test_2021_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

salida.head()

,fecha,generacion_real,generacion_predicha,error,modelo
0,2025-01-01,78619.626,81534.011736,-2914.385736,RegresionLineal
1,2025-01-02,92660.075,80888.474520,11771.600480,RegresionLineal
2,2025-01-03,75315.585,88973.194669,-13657.609669,RegresionLineal
3,2025-01-04,70088.858,76850.484383,-6761.626383,RegresionLineal
4,2025-01-05,40197.193,74744.315205,-34547.122205,RegresionLineal


## 12. Importancia de variables

Solo aplicable a los modelos basados en árboles (Random Forest / Gradient Boosting).

In [ ]:
importancias = pd.DataFrame({
    "feature": features,
    "coeficiente": modelo_elegido.coef_,
    "std_feature": X_train.std().values
})


importancias["importancia_estandarizada"] = (
    importancias["coeficiente"] * importancias["std_feature"]
).abs()

importancias = importancias.sort_values("importancia_estandarizada", ascending=False)
importancias

,feature,coeficiente,std_feature,importancia_estandarizada
0,generacion_lag_1,0.721527,35639.242996,25714.673561
5,tmax_lag_1,-1547.377814,7.825830,12109.516074
4,tmin_lag_1,1739.116803,5.851690,10176.772466
2,generacion_media_7d,0.302566,33439.400905,10117.631213
10,presMax_lag_1,468.942319,5.121363,2401.624044
15,dia_año,-16.538424,104.678206,1731.212577
8,racha_lag_1,799.857399,2.100567,1680.153785
7,velmedia_lag_1,-1432.386786,0.746685,1069.541194
9,hrMedia_lag_1,-73.288421,14.463465,1060.004503
11,presMin_lag_1,-177.693414,5.539387,984.312518


## 13. Exportación para PowerBI

In [16]:
# Exportar tabla de metricas de comparacion de modelos
tabla_resultados.to_csv(
    "../data/gold/powerbi_metricas_modelos.csv",
    index=False, encoding="utf-8-sig"
)

# Exportar errores por mes del modelo elegido
errores_por_mes.reset_index().to_csv(
    "../data/gold/powerbi_errores_por_mes.csv",
    index=False, encoding="utf-8-sig"
)

# Exportar resultados del backtesting
tabla_backtest.to_csv(
    "../data/gold/powerbi_backtesting.csv",
    index=False, encoding="utf-8-sig"
)

print("Exportado: powerbi_metricas_modelos.csv, powerbi_errores_por_mes.csv, powerbi_backtesting.csv")
print("Ya disponible de la seccion 10: predicciones_test_2021_2025.csv")

Exportado: powerbi_metricas_modelos.csv, powerbi_errores_por_mes.csv, powerbi_backtesting.csv
Ya disponible de la seccion 10: predicciones_test_2021_2025.csv
